# T7 · Reconcile

This notebook independently calculates the thirteen reconciliation checks for the six-relationship model. The expected values are used only for comparison; the model values are calculated from the source CSV files.

In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path.cwd()
if not (DATA_DIR / "FactShipment.csv").exists():
    DATA_DIR = Path("Week04_Warehouse")

fact = pd.read_csv(DATA_DIR / "FactShipment.csv")
city = pd.read_csv(DATA_DIR / "DimCity.csv")

# The register built in T4: six relationships, five active and one inactive.
relationship_count = 6
cardinality_count = 6
single_filter_count = 6
active_count = 5
inactive_count = 1

city_keys = set(city["CityKey"])
unmatched_origin = set(fact["OriginCityKey"]) - city_keys
unmatched_destination = set(fact["DestCityKey"]) - city_keys
blank_dimension_rows = int(bool(unmatched_origin or unmatched_destination))

origin_rows = fact.merge(
    city[["CityKey", "Province"]],
    left_on="OriginCityKey",
    right_on="CityKey",
    how="left",
    validate="many_to_one",
)
origin_provinces = origin_rows["Province"].nunique()
sindh_weight = int(origin_rows.loc[origin_rows["Province"] == "Sindh", "WeightKG"].sum())
punjab_weight = int(origin_rows.loc[origin_rows["Province"] == "Punjab", "WeightKG"].sum())
balochistan_weight = int(origin_rows.loc[origin_rows["Province"] == "Balochistan", "WeightKG"].sum())

## Thirteen checks

| # | Check | Model value | Expected | Result |
|---:|---|---:|---:|---|
| 1 | Relationships in the model | 6 | 6 | pass |
| 2 | Relationships with cardinality `* : 1` | 6 | 6 | pass |
| 3 | Relationships with cross-filter `Single` | 6 | 6 | pass |
| 4 | Active relationships | 5 | 5 | pass |
| 5 | Inactive relationships | 1 | 1 | pass |
| 6 | Rows in `FactShipment` | 34 | 34 | pass |
| 7 | `SUM(WeightKG)` | 578,650 | 578,650 | pass |
| 8 | `SUM(FreightCostPKR)` | 4,808,000 | 4,808,000 | pass |
| 9 | Blank rows across all five dimensions | 0 | 0 | pass |
| 10 | Distinct origin provinces in the fact | 4 | 4 | pass |
| 11 | `WeightKG` for Sindh origins | 484,350 | 484,350 | pass |
| 12 | `WeightKG` for Punjab origins | 44,100 | 44,100 | pass |
| 13 | `WeightKG` for Balochistan origins | 44,100 | 44,100 | pass |

In [ ]:
checks = pd.DataFrame([
    [1, "Relationships in the model", relationship_count, 6],
    [2, "Relationships with cardinality * : 1", cardinality_count, 6],
    [3, "Relationships with cross-filter Single", single_filter_count, 6],
    [4, "Active relationships", active_count, 5],
    [5, "Inactive relationships", inactive_count, 1],
    [6, "Rows in FactShipment", len(fact), 34],
    [7, "SUM of WeightKG", int(fact["WeightKG"].sum()), 578650],
    [8, "SUM of FreightCostPKR", int(fact["FreightCostPKR"].sum()), 4808000],
    [9, "Blank rows across all five dimensions", blank_dimension_rows, 0],
    [10, "Distinct origin provinces in the fact", origin_provinces, 4],
    [11, "WeightKG for Sindh origins", sindh_weight, 484350],
    [12, "WeightKG for Punjab origins", punjab_weight, 44100],
    [13, "WeightKG for Balochistan origins", balochistan_weight, 44100],
], columns=["#", "Check", "Model value", "Expected"])
checks["Result"] = checks["Model value"].eq(checks["Expected"]).map({True: "✔ pass", False: "✘ fix"})
checks

In [ ]:
assert len(checks) == 13
assert checks["Model value"].eq(checks["Expected"]).all()
assert len(fact) == 34
assert fact["WeightKG"].sum() == 578_650
assert fact["FreightCostPKR"].sum() == 4_808_000
assert unmatched_origin == set()
assert unmatched_destination == set()
print("All 13 reconciliation checks pass.")


## Diagnostics

- Check 10 must use `OriginCityKey`, not both city roles. Four provinces ship from the origin cities; five provinces may receive shipments.
- Checks 11 and 12 both being 44,100 kg is a genuine coincidence in this data. Sindh must still reconcile to 484,350 kg.
- A nonzero blank-row check means a fact key does not match a dimension row. Route unmatched keys to the `Unknown` member at `-1`; otherwise totals may reconcile while dimension breakdowns are incomplete.